In [ ]:
%pip -q install hmmlearn scikit-learn pandas numpy matplotlib

# RegDet V1.1 — **v6 → v7 SIDEWAYS ABLATION**

The user identified **v6** as the best version this project produced. Between v6
and v7 SIDEWAYS occupancy TRIPLED on the user's real 2h data:

| run | SIDEWAYS | transition warnings |
|---|---|---|
| `v6_slowed` | **10.6 %** | 92 |
| `v7 (12ka3)` | **36.3 %** | 194 |

Seven config items differ. This notebook changes **exactly one at a time** from the
v6 baseline and measures the result, so the jump can be attributed rather than
guessed at.

| arm | one change from v6 |
|---|---|
| **A0** | v6 BASELINE — `K=4`, `w=0.50`, `frozen_z`, `rank`, `allow` |
| **A1** | `+ [5] INTENSITY_MODE='vol_norm'` |
| **A2** | `+ [6] DIRECTION_MODE='rank'` (the stated prime suspect) |
| **A3** | `+ [7] ESCALATION_DURING_HOLD='block'` |
| **A4** | `+ BAR_DIR_WEIGHT 0.50 → 0.75` |
| **A5** | `+ ENSEMBLE_K 4 → 6` |
| **A6** | **FULL v7** — all of the above stacked. **This is the control.** |
| **C1** | v6 baseline + `BAR_DIR_WEIGHT=0.75` (the combination nobody has run) |
| **C2** | C1 + whichever of [5]/[7] proved harmless |

**A6 is the validity check for the whole exercise.** If stacking every change does
not move SIDEWAYS in the same direction and of a comparable size to the real
v6 → v7 jump, then the ablation is not capturing the real difference and no arm
in between means anything. That is stated as a pass/fail at the end, not glossed.

---

## ⚠ DATA PROVENANCE AND WHAT DOES AND DOES NOT TRANSFER

> ### `GITHUB DATA — UNVERIFIED PROVENANCE, machinery-grade not decision-grade`
>
> `yfinance` and `stooq` are proxy-BLOCKED (403) in this environment. The price
> series used here is **daily** NIFTY 50 OHLC pulled from a third-party GitHub
> repository (`NupurBachhuka/financial_forecast_v2`). It is re-sanity-checked in
> §2 below, but its provenance is **not** verified against an exchange feed.
>
> **It is DAILY. The user's numbers are 2h.** A daily bar carries several times
> the per-bar noise of a 2h bar, so the ABSOLUTE SIDEWAYS levels here will not
> match 10.6 % / 36.3 % and are not meant to.
>
> **What must transfer is the RANKING of the arms and the SIZE of the jump
> between them.** Read the deltas, not the levels. A second, secondary table is
> run on the master's own SYNTHETIC 2h generator purely to check that the size of
> the jump is bar-frequency-robust — that series is synthetic and is illustrative
> only.
>
> **The user's Kaggle 2h run remains the source of truth.** Nothing here is a
> decision; it is an attribution.

---

**Provenance of every line of engine code below:** ENGINE / CONSTANTS / PLOT / FIDELITY: build_master_notebook_v2.py cells 3, 7, 9, 76 (harvested verbatim). ZIGZAG + S/L/W + GUARDS G1-G4 + contiguous band painter: build_stability_lag.py (harvested verbatim).

Nothing in this notebook imports, modifies or re-derives the repo,
`regdet_v11_master.ipynb`, `build_master_notebook_v2.py`, `regime_scorecard.py` or
any sibling generator.

## 1. Engine — harvested VERBATIM, not re-derived

Four cells follow, each inlined byte-for-byte out of the master generator:
constants, the feature + labeling engine (including `label_bars_legacy`, the
FROZEN pre-change copy used as the A0 anchor), the plot helpers, and the
descriptive-fidelity metric.

Causality is inherited from the master engine, not re-argued here: filtered
(forward-only) posteriors, a fit-window-frozen trend baseline, a causal
confirmation delay, and a left-to-right gate state machine. No signal, filter or
label at bar *t* reads a bar > *t*.

In [ ]:
# ==========================================================================
# CONSTANTS -- INLINED VERBATIM (build_master_notebook_v2.py code cell 3)
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier (1.0 = production windows)

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = 9      # bars over which efficiency is measured (~ TREND_FEATURE horizon)
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.0 -- FIX 4 IS RETAINED AS A SWITCH BUT SET OFF.
#
# It was 0.75 (and before that 0.5). It is now 0.0. The machinery, the sweep in
# 7H-vi and the overlay in 7H-vii all stay; only the shipped weight moved.
#
# WHY IT WAS TURNED OFF. Scored across three real-data runs, the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.0
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=6 was the size the
# offline study measured (direction-call agreement between independent pools
# 81.9% single -> 91.9% at K=6), and K=6 is now the ADOPTED value: it is the size
# the stability study actually measured, and the cost is linear. The earlier K=4
# compromise existed only because Section 5d re-ran the ENTIRE walk-forward in
# both arms; 5d is OFF by default now (RUN_SEED_STABILITY=False below), so the
# runtime argument for K=4 no longer applies.
ENSEMBLE_K     = 6
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5, VOL_SLOW=20, SWING_WIN=20)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'vol_norm'    # 'frozen_z' = pre-change | 'vol_norm' = adopted
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'block'   # 'block' (adopted) | 'allow' (pre-change) | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON. NOTE: w={BAR_DIR_WEIGHT} is NOT the shipped "
          "default (0.0); it broke")
    print("              the direction-level forward-return ordering on real data.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# ENGINE -- INLINED VERBATIM (build_master_notebook_v2.py code cell 7)
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    n_side = max(1, round(n_st / 5))
    n_bear = (n_st - n_side) // 2
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# PLOT HELPERS -- INLINED VERBATIM (build_master_notebook_v2.py code cell 9)
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i - 1]))
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

In [ ]:
# ==========================================================================
# DESCRIPTIVE FIDELITY -- INLINED VERBATIM (master code cell 76),
# the master w-sweep SECONDARY metric. It is DESCRIPTION, not prediction, and
# must never be quoted as validation. It is reported because the user values it:
# it is what they liked about v7 (0.886 at w=0.75 vs 0.710 at w=0.0), so a fix
# that restores low SIDEWAYS while destroying fidelity is not a win.
# ==========================================================================
W_SECONDARY_K = 9             # headline trailing lookback, ~3 days of 2h bars
W_SECONDARY_METRIC_NAME = ('fraction of non-SIDEWAYS bars where sign(direction label) '
                           '== sign(C[t] - C[t-k]), k=%d, STRICTLY CAUSAL' % W_SECONDARY_K)


def W_SECONDARY_PERBAR(labels, close_arr, k=None):
    # Per-bar agreement between the emitted direction and the TRAILING move.
    # Returns (agree_bool_array, scored_mask), both length n.
    #
    # STRICTLY CAUSAL BY CONSTRUCTION: entry t reads labels[t], close[t] and
    # close[t-k] -- indices <= t only. Nothing in this function indexes forward.
    k = W_SECONDARY_K if k is None else int(k)
    labels = np.asarray(labels, dtype=object)
    close_arr = np.asarray(close_arr, dtype=float)
    n = len(close_arr)
    lab_sign = np.where(np.isin(labels, ['H_BULL', 'L_BULL']), 1.0,
                        np.where(np.isin(labels, ['H_BEAR', 'L_BEAR']), -1.0, 0.0))
    trail = np.full(n, np.nan)
    if k < n:
        trail[k:] = close_arr[k:] - close_arr[:n - k]   # C[t] - C[t-k]
    move_sign = np.sign(trail)
    m = (lab_sign != 0) & np.isfinite(move_sign) & (move_sign != 0)
    return (lab_sign == move_sign), m


def W_SECONDARY_METRIC(labels, close_arr, k=None):
    # DESCRIPTIVE FIDELITY -- how well the labels describe the move that has
    # ALREADY HAPPENED. This is NOT predictive skill and must never be quoted as
    # validation; see the prose above for why it is partly mechanical at high w.
    #
    # SIDEWAYS bars are excluded (no directional claim); bars t < k are excluded
    # (no lookback available yet).
    #
    # Returns (agreement_fraction, n_scored).
    agree, m = W_SECONDARY_PERBAR(labels, close_arr, k)
    if m.sum() == 0:
        return np.nan, 0
    return float(np.mean(agree[m])), int(m.sum())

print('descriptive fidelity metric ready:')
print(' ', W_SECONDARY_METRIC_NAME)

## 2. Data — GitHub daily NIFTY 50, re-sanity-checked here

`GITHUB DATA — UNVERIFIED PROVENANCE, machinery-grade not decision-grade.`

The file is fetched fresh if the network allows and falls back to the local copy
otherwise; either way the checks below are re-run on whatever is loaded, in this
notebook, rather than trusted from a prior agent's report.

**The engine needs a VIX series** for the `vix_chg` feature and for the transition
warning flag. This file has no India VIX, so a **causal realised-vol proxy**
(20-bar trailing annualised sigma) is substituted. That is declared, not hidden,
and it is identical across every arm, so it cannot bias a between-arm comparison.

In [ ]:
# ===========================================================================
# DATA -- REAL DAILY NIFTY 50 OHLC.  GITHUB DATA - UNVERIFIED PROVENANCE.
# ===========================================================================
import os, io, contextlib, warnings, time
warnings.filterwarnings('ignore')

CSV_URL = ('https://raw.githubusercontent.com/NupurBachhuka/financial_forecast_v2/'
           'main/rawdata/NIFTY50_2015_2026_merged.csv')
CSV_LOCAL = 'nifty_daily.csv'
DATA_TAG = 'GITHUB DATA - UNVERIFIED PROVENANCE, machinery-grade not decision-grade'

_raw_txt, _how = None, None
try:
    import urllib.request
    with urllib.request.urlopen(CSV_URL, timeout=60) as _r:
        _raw_txt = _r.read().decode()
    _how = 'downloaded fresh from GitHub'
    with open(CSV_LOCAL, 'w') as _f:
        _f.write(_raw_txt)
except Exception as _e:                                    # noqa: BLE001
    print(f'  download failed ({type(_e).__name__}: {_e}); falling back to local copy')
    for _cand in (CSV_LOCAL, f'{os.getcwd()}/_sw/nifty_daily.csv'):
        if os.path.exists(_cand):
            _raw_txt = open(_cand).read()
            _how = f'read from local cache {_cand}'
            break
assert _raw_txt is not None, 'no NIFTY daily CSV available (network AND cache failed)'

d = pd.read_csv(io.StringIO(_raw_txt))
d['Date'] = pd.to_datetime(d['Date'])
d = d.sort_values('Date').reset_index(drop=True)

# ---- SANITY CHECKS, RE-RUN HERE (not inherited from a prior report) --------
assert list(d.columns[:6]) == ['Index Name', 'Date', 'Open', 'High', 'Low', 'Close']
assert d['Date'].is_monotonic_increasing, 'dates not monotonic'
assert not d['Date'].duplicated().any(), 'duplicate dates'
assert (d[['Open', 'High', 'Low', 'Close']] > 0).all().all(), 'non-positive price'
assert (d['High'] >= d['Low']).all(), 'High < Low'
assert (d['High'] >= d[['Open', 'Close']].max(axis=1)).all(), 'High below O/C'
assert (d['Low'] <= d[['Open', 'Close']].min(axis=1)).all(), 'Low above O/C'
_r = np.log(d['Close'] / d['Close'].shift(1)).dropna()
assert _r.abs().max() < 0.20, f'implausible daily move {_r.abs().max():.3f}'

nifty = pd.Series(d['Close'].values, index=pd.DatetimeIndex(d['Date']), name='nifty')

# Causal realised-vol VIX proxy. DECLARED, not hidden. Identical across all arms.
_rr = np.log(nifty / nifty.shift(1))
vix = (_rr.rolling(20).std() * np.sqrt(252) * 100).bfill().clip(6, 90)
vix.name = 'vix'
TAC_SYNTH = False

print('=' * 100)
print(f'*** {DATA_TAG} ***')
print('=' * 100)
print(f'  source     : {CSV_URL}')
print(f'  loaded     : {_how}')
print(f'  rows       : {len(d)}   span {nifty.index[0]:%Y-%m-%d} -> {nifty.index[-1]:%Y-%m-%d}')
print(f'  bar        : DAILY  (the user\'s reference runs are 2h -- absolute SIDEWAYS')
print( '               levels will NOT match; the RANKING and the SIZE of the jump')
print( '               between arms are what must transfer)')
print(f'  vix        : REALISED-VOL PROXY (20-bar trailing annualised sigma), not India VIX')
print(f'  OHLC / monotonicity / no-duplicate / max |daily move| checks: ALL PASS'
      f'  (max |r| = {_r.abs().max():.4f})')

# ---- known-event spot checks ----------------------------------------------
def _px(day):
    s = nifty.loc[:day]
    return float(s.iloc[-1]) if len(s) else float('nan')

_events = [('2020-03-23', 'COVID crash low',   7610.0, 0.03),
           ('2015-01-01', 'series start',      8284.0, 0.01),
           ('2025-05-09', 'May-2025 V low',   24008.0, 0.03)]
print('  known-event spot checks (tolerance shown):')
for _dt, _what, _exp, _tol in _events:
    _got = _px(_dt)
    _ok = abs(_got - _exp) / _exp <= _tol
    print(f'    {_dt}  {_what:20s} expect~{_exp:9.1f}  got {_got:9.1f}  '
          f'{"OK" if _ok else "*** OFF ***"}')
    assert _ok, f'{_what} spot check failed'
print('  -> the series is the real NIFTY 50 by behaviour. Provenance is still UNVERIFIED.')

## 3. Metrics — S / L / W and guards G1–G4, harvested VERBATIM

Sliced byte-for-byte out of `build_stability_lag.py` so the guards that void an
arm here are the *same code* that voided arms there.

| | |
|---|---|
| **S** | mean pairwise 5-label agreement across R = 4 **disjoint** seed sets |
| **L** | bars from a ZigZag swing start to the first correctly-directed emitted label |
| **W** | 5-label switches per 100 bars — the anti-gaming guard on S |

`G1` every label occupancy in [3 %, 50 %] · `G2` no label collapses below 3 % ·
`G3` W ≤ 25 / 100 bars · `G4` SIDEWAYS ≤ 50 %.

**G4 exists because S is trivially maximised by labelling everything SIDEWAYS.**
§4 drives that exact degenerate case through the real guard code and asserts it is
VOIDED — the guard is tested, not trusted.

The **ZigZag is retrospective and label-blind**: it sees prices only, and it is
computed only AFTER every label in this notebook already exists. A tripwire
shadows `label_bars` and asserts that ordering, plus object identity and memory
aliasing against every ZigZag output.

In [ ]:
# ==========================================================================
# METRIC PARAMETERS -- ALL DECLARED HERE, BEFORE ANY NUMBER EXISTS.
# (values identical to build_stability_lag.py / STABILITY_LAG_PROTOCOL.md)
# ==========================================================================
R_SEED_SETS   = 4       # protocol: R = 4 DISJOINT seed sets. NOT reducible.
ZZ_PCT        = 2.0     # protocol: ZigZag reversal threshold, 2.0%
W_GUARD_MAX   = 25.0    # G3
OCC_MIN_PCT   = 3.0     # G1 / G2
OCC_MAX_PCT   = 50.0    # G1
SIDEWAYS_MAX  = 50.0    # G4

print(f'R = {R_SEED_SETS} disjoint seed sets   ZigZag = {ZZ_PCT}%   '
      f'G3 W<={W_GUARD_MAX}   G1 occ in [{OCC_MIN_PCT},{OCC_MAX_PCT}]%   '
      f'G4 SIDEWAYS<={SIDEWAYS_MAX}%')

In [ ]:
# ==========================================================================
# THE ZIGZAG -- INLINED VERBATIM from build_stability_lag.py.
# LABEL-BLIND, RETROSPECTIVE. Sees prices only; never a label.
# ==========================================================================
ZZ_CALLS = 0            # how many times a ZigZag has been computed
ZZ_IDS = set()          # id() of every object a ZigZag has ever returned
ZZ_ARRAYS = []          # the arrays themselves, for np.shares_memory checks


def zigzag_pivots(px, pct=ZZ_PCT):
    '''Indices of ZigZag pivots on a close series, `pct`% reversal threshold.

    RETROSPECTIVE BY CONSTRUCTION: a pivot at bar i is only confirmed once price
    has moved pct% away from it, which happens at some bar > i. That is exactly
    the look-ahead this function is allowed to have and `label_bars` is not.

    Takes a bare float array in and returns a bare int array out -- it has no
    access to a label, a mass, a model or a feature frame.
    '''
    global ZZ_CALLS
    px = np.asarray(px, dtype=float)
    n = len(px)
    if n < 3:
        out = np.array([], dtype=int)
    else:
        hi_i = lo_i = 0
        d, start, piv, ext_i = 0, None, [], 0
        for i in range(1, n):
            if px[i] > px[hi_i]:
                hi_i = i
            if px[i] < px[lo_i]:
                lo_i = i
            if (px[hi_i] - px[i]) / px[hi_i] * 100.0 >= pct:
                d, piv, ext_i, start = -1, [hi_i], i, i
                break
            if (px[i] - px[lo_i]) / px[lo_i] * 100.0 >= pct:
                d, piv, ext_i, start = +1, [lo_i], i, i
                break
        if d == 0:
            out = np.array([], dtype=int)
        else:
            for i in range(start + 1, n):
                if d > 0:
                    if px[i] >= px[ext_i]:
                        ext_i = i
                    elif (px[ext_i] - px[i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = -1, i
                else:
                    if px[i] <= px[ext_i]:
                        ext_i = i
                    elif (px[i] - px[ext_i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = +1, i
            if ext_i != piv[-1]:
                piv.append(ext_i)          # final, PROVISIONAL pivot
            out = np.asarray(piv, dtype=int)

    ZZ_CALLS += 1
    ZZ_IDS.add(id(out))
    ZZ_ARRAYS.append(out)
    return out


def zigzag_swings(px, pct=ZZ_PCT):
    '''[(i0, i1, +1/-1), ...] -- consecutive pivot pairs that clear `pct`.

    The last (provisional) leg is kept only if it actually cleared the threshold,
    so a half-formed leg at the right edge cannot inflate or deflate lag.
    '''
    px = np.asarray(px, dtype=float)
    piv = zigzag_pivots(px, pct)
    sw = []
    for a, b in zip(piv[:-1], piv[1:]):
        move = (px[b] - px[a]) / px[a] * 100.0
        if abs(move) >= pct:
            sw.append((int(a), int(b), 1 if move > 0 else -1))
    out = sw
    ZZ_IDS.add(id(out))
    return out

In [ ]:
# ==========================================================================
# S / L / W + GUARDS G1..G4 -- INLINED VERBATIM from build_stability_lag.py.
# ==========================================================================
DIR_OF_LABEL = {'H_BULL': 1, 'L_BULL': 1, 'SIDEWAYS': 0, 'L_BEAR': -1, 'H_BEAR': -1}


def emitted_direction(labels):
    '''5-label string array -> +1 bull / 0 sideways / -1 bear.'''
    return np.array([DIR_OF_LABEL[x] for x in np.asarray(labels)], dtype=int)


def metric_S(label_sets):
    '''S = mean pairwise 5-LABEL agreement (%) across R seed sets.

    Returns (S_mean, pairwise_matrix RxR, list_of_pairwise_values).
    The FULL matrix is returned because the protocol requires it reported, not
    just the mean.
    '''
    R = len(label_sets)
    M = np.full((R, R), np.nan)
    vals = []
    for i in range(R):
        M[i, i] = 100.0
        for j in range(i + 1, R):
            a = np.asarray(label_sets[i]); b = np.asarray(label_sets[j])
            assert len(a) == len(b), 'label sets must be the same length'
            v = 100.0 * float(np.mean(a == b))
            M[i, j] = M[j, i] = v
            vals.append(v)
    return float(np.mean(vals)), M, vals


def metric_L(labels, swings):
    '''L per ZigZag swing = bars from swing START to the first correctly
    directed EMITTED label inside the swing.

    An UNMATCHED swing (never labelled correctly anywhere inside it) scores the
    FULL swing length -- the worst case, per protocol. It is not dropped.

    Returns (median, p75, per_swing_array, n_unmatched).
    '''
    d = emitted_direction(labels)
    lags, unmatched = [], 0
    for i0, i1, sgn in swings:
        seg = d[i0:i1 + 1]
        hit = np.flatnonzero(seg == sgn)
        if hit.size:
            lags.append(int(hit[0]))
        else:
            lags.append(int(i1 - i0))
            unmatched += 1
    if not lags:
        return np.nan, np.nan, np.array([]), 0
    a = np.asarray(lags, dtype=float)
    return float(np.median(a)), float(np.percentile(a, 75)), a, unmatched


def metric_W(labels):
    '''W = 5-label switches per 100 bars.'''
    a = np.asarray(labels)
    if len(a) < 2:
        return 0.0
    return 100.0 * float(np.sum(a[1:] != a[:-1])) / (len(a) - 1)


def occupancy_pct(labels):
    a = np.asarray(labels)
    return {lb: 100.0 * float(np.mean(a == lb)) for lb in REGIME_LABELS}


# ---------------------------------------------------------------------------
# GUARDS G1..G4. One implementation, used by real arms AND by the degeneracy
# stub in section 3, so the test exercises the shipping code path.
# ---------------------------------------------------------------------------
def evaluate_guards(occ, W):
    '''{name: (bool_pass, reason)} for the four protocol guards.'''
    g = {}
    bad_hi = [l for l in REGIME_LABELS if occ.get(l, 0.0) > OCC_MAX_PCT]
    bad_lo = [l for l in REGIME_LABELS if occ.get(l, 0.0) < OCC_MIN_PCT]
    g['G1'] = (not bad_hi and not bad_lo,
               'ok' if not (bad_hi or bad_lo)
               else f'outside [{OCC_MIN_PCT},{OCC_MAX_PCT}]%: '
                    + ','.join(sorted(set(bad_hi + bad_lo))))
    g['G2'] = (not bad_lo, 'ok' if not bad_lo else 'collapsed: ' + ','.join(bad_lo))
    g['G3'] = (W <= W_GUARD_MAX, f'W={W:.2f} vs max {W_GUARD_MAX}')
    g['G4'] = (occ.get('SIDEWAYS', 0.0) <= SIDEWAYS_MAX,
               f"SIDEWAYS={occ.get('SIDEWAYS', 0.0):.1f}% vs max {SIDEWAYS_MAX}%")
    return g


def guards_pass(g):
    return all(v[0] for v in g.values())

print('metrics ready: metric_S (pairwise matrix), metric_L (median/p75, unmatched '
      '= full swing length), metric_W, evaluate_guards (G1-G4).')
print('NO composite score is defined anywhere in this notebook -- by design.')

In [ ]:
# ===========================================================================
# THE LEAKAGE TRIPWIRE. `label_bars` is SHADOWED by a guard; every later call in
# this notebook goes through it.
#
# ADAPTED from build_stability_lag.py in EXACTLY ONE respect, and it is a
# deliberate, declared deviation: that notebook decreed BAR_DIR_WEIGHT == 0.0 on
# every call, because w was frozen for that study. THIS study VARIES w (0.50 vs
# 0.75) -- that is arm A4 / C1. Check 4 is therefore re-pointed at the ARM'S
# DECLARED WEIGHT: every call must use exactly the weight the arm table declares
# for the arm currently being run, so w still cannot drift silently inside a
# sweep. Checks 1, 2 and 3 (phase ordering, object identity, memory aliasing) are
# unchanged and stay armed throughout.
# ===========================================================================
import contextlib

_label_bars_raw = label_bars
LABEL_CALLS = 0
LABEL_PHASE_OPEN = True     # flipped shut once the EVAL phase starts
CURRENT_ARM = None          # (name, declared_bar_dir_weight); set by the runner


@contextlib.contextmanager
def running_arm(name, w):
    global CURRENT_ARM
    CURRENT_ARM = (name, float(w))
    try:
        yield
    finally:
        CURRENT_ARM = None


def label_bars(*args, **kwargs):
    '''Guarded shadow of the engine's label_bars. Adds asserts, changes nothing.'''
    global LABEL_CALLS
    # check 1 -- PHASE ORDERING: no label may be produced after a ZigZag exists.
    assert ZZ_CALLS == 0 or LABEL_PHASE_OPEN, (
        'LEAKAGE: a label was produced AFTER a ZigZag had been computed. All '
        'labelling must complete in the LABEL PHASE, before any ZigZag exists.')
    # check 4 -- the arm's DECLARED weight, not a global constant.
    _bw = kwargs.get('bar_dir_weight', BAR_DIR_WEIGHT)
    assert CURRENT_ARM is not None, 'label_bars called outside running_arm()'
    assert _bw == CURRENT_ARM[1], (
        f'BAR_DIR_WEIGHT drifted to {_bw} inside arm {CURRENT_ARM[0]!r} '
        f'(declared {CURRENT_ARM[1]})')
    # checks 2 & 3 -- object identity and memory aliasing vs every ZigZag output.
    for v in list(args) + list(kwargs.values()):
        assert id(v) not in ZZ_IDS, 'LEAKAGE: a ZigZag output was passed to label_bars'
        if isinstance(v, np.ndarray):
            for z in ZZ_ARRAYS:
                assert not np.shares_memory(v, z), \
                    'LEAKAGE: a label_bars argument aliases ZigZag memory'
    LABEL_CALLS += 1
    return _label_bars_raw(*args, **kwargs)


print('leakage tripwire armed on label_bars:')
print('  check 1  phase ordering  : label_bars asserts ZZ_CALLS == 0')
print('  check 2  object identity : every argument checked against ZZ_IDS')
print('  check 3  memory aliasing : np.shares_memory vs every ZigZag array')
print('  check 4  declared weight : bar_dir_weight == the ARM TABLE value (adapted)')

## 4. Machinery self-tests — including **driving G4 with a degenerate stub**

1. **`confirm_delay(raw, 1) is raw`** — the off-switch is exact.
2. **G4 bites.** A stub that forces an all-`SIDEWAYS` labelling is pushed through
   the *same* summarising and guard code the real arms use. It scores a perfect
   `S = 100 %` and must still come out VOID.
3. **G3 bites** on a synthetic high-whipsaw labelling.

The ZigZag's own self-test lives in §8, *after* the label phase — computing a
ZigZag here would set `ZZ_CALLS` non-zero and make the label phase's ordering
report untrue.

In [ ]:
# ===========================================================================
# SELF-TESTS. These run BEFORE any arm, against the REAL guard path.
# ===========================================================================
# ---- TEST 1: confirm_delay(raw, 1) is the identity -------------------------
_rng = np.random.default_rng(0)
_probe = _rng.choice(np.array(['BULL', 'BEAR', 'SIDE']), 500)
assert (confirm_delay(_probe, 1) == _probe).all(), 'confirm_delay(.,1) is not the identity'
print('TEST 1 PASS  confirm_delay(raw, confirm_bars=1) reproduces raw exactly.')

# NOTE: the ZigZag self-test is deliberately NOT here. It is in section 8, after
# the label phase closes -- computing a ZigZag at this point would make ZZ_CALLS
# non-zero and the label phase's ordering report untrue.
assert ZZ_CALLS == 0, 'a ZigZag was computed before the label phase'

# ---- the shared summariser, used by REAL arms AND by the degenerate stubs ---
def summarise(label_sets, close_arr, swings=None):
    '''S, L, W, occupancy and guards TOGETHER, from one function, so the guard
    test exercises the shipping path rather than a copy of it.'''
    S, Smat, _ = metric_S(label_sets)
    lab0 = np.asarray(label_sets[0])
    W = metric_W(lab0)
    occ = occupancy_pct(lab0)
    if swings is None:
        L = Lp75 = np.nan; nun = 0
    else:
        L, Lp75, _la, nun = metric_L(lab0, swings)
    g = evaluate_guards(occ, W)
    fid, nfid = W_SECONDARY_METRIC(lab0, close_arr)
    return dict(S=S, S_matrix=Smat, L=L, L_p75=Lp75, L_unmatched=nun, W=W,
                occ=occ, guards=g, void=not guards_pass(g),
                fidelity=fid, fidelity_n=nfid)


# ---- TEST 3: G4 MUST BITE on a degenerate all-SIDEWAYS labelling -----------
_n_probe = 600
_all_side = [np.full(_n_probe, 'SIDEWAYS', dtype='<U8') for _ in range(R_SEED_SETS)]
_r4 = summarise(_all_side, np.asarray(nifty.values[:_n_probe], dtype=float))
print(f"\nDEGENERATE STUB (every bar forced SIDEWAYS): S={_r4['S']:.2f}%  W={_r4['W']:.2f}")
for _k4, (_p, _why) in _r4['guards'].items():
    print(f"    {_k4}: {'PASS' if _p else 'FAIL'}   {_why}")
assert abs(_r4['S'] - 100.0) < 1e-9, 'the degenerate stub should score a perfect S'
assert not _r4['guards']['G4'][0], 'G4 FAILED TO BITE on an all-SIDEWAYS labelling'
assert _r4['void'], 'the degenerate stub was not VOIDED'
print('TEST 2 PASS  a perfect S=100.00% is VOIDED by G4 (and G1/G2). '
      'The anti-gaming guard is TESTED, not trusted.')

# ---- TEST 4: G3 MUST BITE on a high-whipsaw labelling ----------------------
_flip = np.array(['H_BULL', 'H_BEAR'] * (_n_probe // 2), dtype='<U8')
_r3 = summarise([_flip] * R_SEED_SETS, np.asarray(nifty.values[:_n_probe], dtype=float))
assert not _r3['guards']['G3'][0], 'G3 FAILED TO BITE at W=100/100 bars'
print(f"TEST 3 PASS  W={_r3['W']:.1f}/100 bars is VOIDED by G3 "
      f"({_r3['guards']['G3'][1]}).")
assert ZZ_CALLS == 0, 'the self-tests must not compute a ZigZag'

## 5. The arms — **declared here, before a single number exists**

Every arm is the v6 baseline with **exactly one** knob moved, except A6 (all of
them) and C1/C2 (the combination arms the user asked for).

The v6 baseline, from the reference run's own printed header:

```
[1] DIRECTION_EXCLUDE=('vol_2h','vol_expansion')
[2] CONFIRM_BARS=2
[3] gate bands enter(|z|>=0.5, eff>=0.35) exit(|z|<0.35, eff<0.25)
[4] BAR_DIR_WEIGHT=0.5   ENSEMBLE_K=4   BASE_SEED=42
    INTENSITY_MODE='frozen_z'   DIRECTION_MODE='rank'   ESCALATION_DURING_HOLD='allow'
```

The last line is what v6's `label_bars` *did*; it had no such switches, and the
master engine documents `frozen_z` / `rank` / `allow` as reproducing the
pre-change behaviour **bit for bit**. §7 proves that rather than assuming it.

`[1]`, `[2]` and `[3]` are held FIXED at their v6 values in every arm — they are
identical in v6 and v7 and are not part of the difference under investigation.

In [ ]:
# ===========================================================================
# THE ARM TABLE -- DECLARED BEFORE ANY RESULT.
# ===========================================================================
V6_BASELINE = dict(
    exclude=DIRECTION_EXCLUDE,          # [1] identical in v6 and v7
    confirm_bars=2,                     # [2] identical in v6 and v7
    z_exit=Z_HI_EXIT, eff_exit=EFF_HI_EXIT,   # [3] identical in v6 and v7
    bar_dir_weight=0.50,                # [4] v6 value
    intensity_mode='frozen_z',          # [5] OFF  (v6 had no such switch)
    direction_mode='rank',              # [6] v6 behaviour
    escalation_during_hold='allow',     # [7] OFF  (v6 had no such switch)
)
V6_K = 4                                # [4] ENSEMBLE_K, v6 value
V7_K = 6

ARMS = [
    ('A0  v6 BASELINE',        V6_K, {},                                          'the anchor'),
    ('A1  +[5] vol_norm',      V6_K, dict(intensity_mode='vol_norm'),             'intensity gate mode'),
    ('A2  +[6] rank',          V6_K, dict(direction_mode='rank'),                 'the stated prime suspect'),
    ('A3  +[7] block',         V6_K, dict(escalation_during_hold='block'),        'escalation-during-hold'),
    ('A4  +w 0.50->0.75',      V6_K, dict(bar_dir_weight=0.75),                   'per-bar direction weight'),
    ('A5  +K 4->6',            V7_K, {},                                          'ensemble size'),
    ('A6  FULL v7',            V7_K, dict(intensity_mode='vol_norm',
                                          direction_mode='rank',
                                          escalation_during_hold='block',
                                          bar_dir_weight=0.75),                   'THE CONTROL'),
    ('C1  v6 + w=0.75',        V6_K, dict(bar_dir_weight=0.75),                   'combination the user asked for'),
]
# C2 is appended in section 8, once the ablation has said which of [5]/[7] is
# harmless. It cannot be declared before that answer exists.

# Reference levels from the user's REAL 2h Kaggle runs. Reported for orientation
# only -- this notebook is DAILY and cannot reproduce these levels.
REF_V6_SIDEWAYS, REF_V6_WARN = 10.6, 92
REF_V7_SIDEWAYS, REF_V7_WARN = 36.3, 194

print('ARMS DECLARED (before any number exists):')
for _nm, _K, _ov, _what in ARMS:
    _cfg = dict(V6_BASELINE); _cfg.update(_ov)
    print(f"  {_nm:22s} K={_K}  w={_cfg['bar_dir_weight']:.2f}  "
          f"{_cfg['intensity_mode']:9s} {_cfg['direction_mode']:5s} "
          f"{_cfg['escalation_during_hold']:6s}   <- {_what}")
print(f'\nreference (user 2h Kaggle, source of truth): '
      f'v6 SIDEWAYS={REF_V6_SIDEWAYS}% warn={REF_V6_WARN}  |  '
      f'v7 SIDEWAYS={REF_V7_SIDEWAYS}% warn={REF_V7_WARN}')

## 6. THE LABEL PHASE

Every label in this notebook is produced here, **before the ZigZag exists**. The
tripwire from §3 enforces that ordering.

Each arm is labelled `R = 4` times, on 4 **disjoint** seed sets, which is what
`S` is measured over. Seed set 0 is the arm's headline labelling and is what every
occupancy / W / fidelity / figure number is taken from — exactly as in
`build_stability_lag.py`.

In [ ]:
# ===========================================================================
# FEATURES, SCALING, FIT WINDOW -- one shared setup, identical for every arm.
# ===========================================================================
from sklearn.preprocessing import StandardScaler

CFG_BY_NAME = {c['name']: c for c in CONFIGS}
EVAL_CFG = CFG_BY_NAME[ADOPTED_CONFIG_NAME]
EVAL_FEAT = tuple(EVAL_CFG['features'])
EVAL_N, EVAL_COV = EVAL_CFG['N'], EVAL_CFG['cov']

FEAT = build_features(nifty, vix, LOOKBACK_SCALE)
DATES = FEAT.index
CLOSE = nifty.reindex(DATES).values.astype(float)
TREND_RAW = FEAT[TREND_FEATURE].values
NBARS = len(DATES)
N_FIT = max(int(NBARS * TRAIN_FRACTION), 50)

_scaler = StandardScaler().fit(FEAT[list(EVAL_FEAT)].values[:N_FIT])
XS = _scaler.transform(FEAT[list(EVAL_FEAT)].values)

print(f'config      : {ADOPTED_CONFIG_NAME}  N={EVAL_N} cov={EVAL_COV} '
      f'{len(EVAL_FEAT)} features')
print(f'bars        : {NBARS}   fit window (anchored, causal): {N_FIT} '
      f'({TRAIN_FRACTION:.0%})   span {DATES[0]:%Y-%m-%d} -> {DATES[-1]:%Y-%m-%d}')
print(f'DATA        : {DATA_TAG}   BAR = DAILY')


def seed_sets(K):
    '''R DISJOINT seed sets of size K: [42..42+K-1], [42+K..], ...'''
    return [[BASE_SEED + s * K + i for i in range(K)] for s in range(R_SEED_SETS)]


FIT_CACHE, FIT_COUNT = {}, 0


def fits(K, base):
    global FIT_COUNT
    key = (K, base)
    if key not in FIT_CACHE:
        with contextlib.redirect_stdout(io.StringIO()):
            ms, conv = fit_hmm_ensemble(XS[:N_FIT], EVAL_N, EVAL_COV,
                                        K=K, base_seed=base)
        FIT_CACHE[key] = ms
        FIT_COUNT += K
    return FIT_CACHE[key]


print(f'seed sets   : R={R_SEED_SETS}  K=4 -> {seed_sets(4)}')
print(f'              R={R_SEED_SETS}  K=6 -> {seed_sets(6)}')

In [ ]:
# ===========================================================================
# LABEL PHASE -- every label in this notebook is produced HERE.
# ===========================================================================
LABELS, FRAMES = {}, {}


def run_arm(name, K, over):
    cfg = dict(V6_BASELINE); cfg.update(over)
    sets = seed_sets(K)
    labs, frames = [], []
    with running_arm(name, cfg['bar_dir_weight']):
        for base in [s[0] for s in sets]:
            ms = fits(K, base)
            with contextlib.redirect_stdout(io.StringIO()):
                out = label_bars(ms, XS, DATES, nifty, TREND_RAW, N_FIT, EVAL_FEAT,
                                 dir_feats=FEAT, feat_raw=FEAT, **cfg)
            labs.append(out['tactical_regime_state'].values)
            frames.append(out)
    LABELS[name] = labs
    FRAMES[name] = frames
    return cfg


ARM_CFG = {}
_t0 = time.time()
for _nm, _K, _ov, _what in ARMS:
    ARM_CFG[_nm] = run_arm(_nm, _K, _ov)
    _s = 100 * np.mean(LABELS[_nm][0] == 'SIDEWAYS')
    print(f'  labelled {_nm:22s} K={_K}  R={R_SEED_SETS}   SIDEWAYS(seed set 0) = {_s:5.2f}%')
print(f'\nLABEL PHASE: {LABEL_CALLS} guarded label_bars calls, {FIT_COUNT} HMM fits, '
      f'{time.time() - _t0:.1f}s.  ZZ_CALLS so far = {ZZ_CALLS} (must be 0 for real arms)')

## 7. THE TWO ANCHORS

Without these, nothing in between means anything.

**Anchor A — A0 really is the v6 direction scheme.** The master engine ships
`label_bars_legacy`, a FROZEN verbatim copy of `label_bars` as it stood *before*
`INTENSITY_MODE` / `DIRECTION_MODE` / `ESCALATION_DURING_HOLD` and the
direction-before-intensity reorder existed — i.e. the v6 function. A0 is run
through both and every emitted column is compared **bit for bit**.

**Anchor B — A6 really is the full v7 stack.** Its resolved config is asserted
field-by-field against the v7 reference run's own printed header.

**Anchor C — the prime suspect, tested directly.** A2 (`DIRECTION_MODE='rank'`)
is compared bit-for-bit against A0.

In [ ]:
# ===========================================================================
# ANCHOR A -- A0 IS THE v6 DIRECTION SCHEME, PROVEN AGAINST THE FROZEN COPY.
# ===========================================================================
_a0 = ARM_CFG['A0  v6 BASELINE']
_legacy_kw = dict(exclude=_a0['exclude'], confirm_bars=_a0['confirm_bars'],
                  z_exit=_a0['z_exit'], eff_exit=_a0['eff_exit'],
                  dir_feats=FEAT, bar_dir_weight=_a0['bar_dir_weight'])
_ms0 = fits(V6_K, seed_sets(V6_K)[0][0])
with contextlib.redirect_stdout(io.StringIO()):
    _leg = label_bars_legacy(_ms0, XS, DATES, nifty, TREND_RAW, N_FIT, EVAL_FEAT,
                             **_legacy_kw)
_new = FRAMES['A0  v6 BASELINE'][0]

_shared = [c for c in _leg.columns if c in _new.columns]
_bad = []
for _c in _shared:
    _x, _y = _leg[_c].values, _new[_c].values
    _same = (np.array_equal(_x, _y) if _x.dtype.kind not in 'fc'
             else np.array_equal(_x, _y, equal_nan=True))
    if not _same:
        _bad.append(_c)
print('ANCHOR A -- A0 vs label_bars_legacy (the FROZEN pre-change / v6 function)')
print(f'  columns compared : {len(_shared)}  ({", ".join(_shared)})')
print(f'  mismatching      : {_bad if _bad else "NONE"}')
assert not _bad, f'A0 is NOT the v6 scheme -- columns differ: {_bad}'
assert (_leg['tactical_regime_state'].values == LABELS['A0  v6 BASELINE'][0]).all()
print('  ==> A0 REPRODUCES THE v6 LABELING FUNCTION BIT FOR BIT. Anchor holds.')

In [ ]:
# ===========================================================================
# ANCHOR B -- A6 IS THE FULL v7 STACK (config asserted against v7's own header).
# ===========================================================================
V7_HEADER = dict(bar_dir_weight=0.75, intensity_mode='vol_norm',
                 direction_mode='rank', escalation_during_hold='block')
V7_HEADER_K = 6
_a6 = ARM_CFG['A6  FULL v7']
print('ANCHOR B -- A6 resolved config vs the v7 reference run printed header')
for _k6, _v6v in V7_HEADER.items():
    print(f'  {_k6:24s} A6={_a6[_k6]!r:12s} v7 header={_v6v!r}   '
          f'{"OK" if _a6[_k6] == _v6v else "*** MISMATCH ***"}')
    assert _a6[_k6] == _v6v, f'A6 does not match the v7 header on {_k6}'
print(f'  {"ENSEMBLE_K":24s} A6={V7_K!r:12s} v7 header={V7_HEADER_K!r}   '
      f'{"OK" if V7_K == V7_HEADER_K else "*** MISMATCH ***"}')
assert V7_K == V7_HEADER_K
# and [1][2][3] are the v6 values, unchanged -- they are identical in v6 and v7.
assert _a6['exclude'] == DIRECTION_EXCLUDE and _a6['confirm_bars'] == 2
assert _a6['z_exit'] == Z_HI_EXIT and _a6['eff_exit'] == EFF_HI_EXIT
print('  ==> A6 IS THE FULL v7 CONFIGURATION. [1][2][3] held at their (shared) v6 values.')

In [ ]:
# ===========================================================================
# ANCHOR C -- THE PRIME SUSPECT, TESTED DIRECTLY.
# ===========================================================================
_identical = bool((np.asarray(LABELS['A0  v6 BASELINE'][0])
                   == np.asarray(LABELS['A2  +[6] rank'][0])).all())
_ndiff = int(np.sum(np.asarray(LABELS['A0  v6 BASELINE'][0])
                    != np.asarray(LABELS['A2  +[6] rank'][0])))
print('ANCHOR C -- A2 (DIRECTION_MODE=\'rank\') vs A0 (v6 baseline)')
print(f'  bars differing : {_ndiff} / {NBARS}')
print(f'  bit-identical  : {_identical}')
print()
print('WHY. Read the harvested engine above: ensemble_direction_masses_by_mode()')
print("  mode='rank' DELEGATES to the unchanged ensemble_direction_masses(), and")
print('  v6\'s direction_buckets() was ALREADY rank-based (bottom 2 bear / middle 1')
print('  side / top 2 bull for N=5). The switchable alternative is \'soft\', which')
print("  v7 implemented but did NOT adopt. So DIRECTION_MODE='rank' is v6's own")
print('  behaviour under a new name: it is a NO-OP between the two versions.')

## 8. Close the label phase, then compute the ZigZag

The ZigZag is retrospective — a pivot at bar *i* is only confirmed once price has
moved 2 % away from it, which happens at some bar > *i*. That is exactly the
look-ahead the ZigZag is allowed and `label_bars` is not, which is why every label
had to exist first.

In [ ]:
# ===========================================================================
# EVAL PHASE OPENS. No further labelling is permitted except inside an explicit,
# LOUD reopening for the C2 combination arm (whose identity could not be known
# until the single interventions had been measured).
# ===========================================================================
LABEL_PHASE_OPEN = False
print('LABEL PHASE CLOSED. label_bars will now assert on any further call.')

# ---- ZigZag self-test: it is LABEL-BLIND. Deferred to here on purpose. -----
_zz_probe = zigzag_pivots(np.asarray(CLOSE[:400], dtype=float), ZZ_PCT)
assert isinstance(_zz_probe, np.ndarray) and _zz_probe.dtype.kind == 'i', \
    'zigzag_pivots must return a bare int index array'
print(f'ZIGZAG SELF-TEST PASS  zigzag_pivots(bare float array) -> bare int array '
      f'({_zz_probe.size} pivots).')
print('  Nothing but prices goes in: it has no access to a label, a mass, a model')
print('  or a feature frame, and it could not have influenced any label above.')

SWINGS = zigzag_swings(CLOSE, ZZ_PCT)
print(f'ZigZag ({ZZ_PCT}% reversal) on {NBARS} DAILY bars -> {len(SWINGS)} swings '
      f'({sum(1 for _, _, s in SWINGS if s > 0)} up / '
      f'{sum(1 for _, _, s in SWINGS if s < 0)} down)')
print(f'ZZ_CALLS = {ZZ_CALLS}  (every real label was produced before this line)')

# The tripwire must now BITE. Proven, not asserted in prose.
try:
    with running_arm('tripwire-probe', V6_BASELINE['bar_dir_weight']):
        label_bars(_ms0, XS, DATES, nifty, TREND_RAW, N_FIT, EVAL_FEAT,
                   dir_feats=FEAT, feat_raw=FEAT, **V6_BASELINE)
    raise SystemExit('TRIPWIRE FAILED TO BITE')
except AssertionError as _e:
    print(f'TRIPWIRE BITES as designed: {str(_e)[:70]}...')

# ZigZag output must never have reached a label. Asserted, not assumed.
assert all(id(np.asarray(l)) not in ZZ_IDS for ls in LABELS.values() for l in ls)
for _ls in LABELS.values():
    for _l in _ls:
        for _z in ZZ_ARRAYS:
            assert not np.shares_memory(np.asarray(_l), _z)
print('ASSERT OK: no ZigZag object or memory ever reached a label.')

In [ ]:
# ===========================================================================
# TRANSITION WARNINGS -- the master's own definition, per arm.
#   causal confidence DROP > HMM_PROB_DROP_THRESHOLD, OR a VIX bar-over-bar spike
#   > VIX_SPIKE_THRESHOLD.  (VIX here is the declared realised-vol proxy.)
# ===========================================================================
def transition_warnings(frame):
    _conf = frame['tactical_regime_confidence']
    _v = pd.Series(vix.reindex(frame.index).values, index=frame.index)
    _twf = ((_conf.shift(1) - _conf) > HMM_PROB_DROP_THRESHOLD) | \
           ((_v - _v.shift(1)) / _v.shift(1) > VIX_SPIKE_THRESHOLD)
    _twf.iloc[0] = False
    return _twf.astype(bool)


print(f'transition warning definition: conf drop > {HMM_PROB_DROP_THRESHOLD} '
      f'OR vix bar-over-bar > {VIX_SPIKE_THRESHOLD:.0%}')

In [ ]:
# ===========================================================================
# C2 -- the second combination arm. Its identity DEPENDS on the ablation result,
# so it is declared HERE, after A1/A3 have been measured, and its labelling
# happens inside an EXPLICIT, LOUD reopening of the label phase.
# ===========================================================================
_side = {nm: 100 * np.mean(LABELS[nm][0] == 'SIDEWAYS') for nm, _, _, _ in ARMS}
_base_side = _side['A0  v6 BASELINE']
HARMLESS = []
if abs(_side['A1  +[5] vol_norm'] - _base_side) < 0.5:
    HARMLESS.append('[5] vol_norm')
if abs(_side['A3  +[7] block'] - _base_side) < 0.5:
    HARMLESS.append('[7] block')
print(f'SIDEWAYS-harmless single interventions (|delta| < 0.5pp vs A0): '
      f'{HARMLESS if HARMLESS else "NONE"}')

_c2_over = dict(bar_dir_weight=0.75)
if '[5] vol_norm' in HARMLESS:
    _c2_over['intensity_mode'] = 'vol_norm'
if '[7] block' in HARMLESS:
    _c2_over['escalation_during_hold'] = 'block'
C2_NAME = 'C2  C1 + ' + (' + '.join(h for h in HARMLESS) if HARMLESS else 'nothing (C1)')

print('!' * 78)
print(f'!! LABEL PHASE EXPLICITLY REOPENED: C2 = {_c2_over}')
print('!! ordering check relaxed; identity / aliasing / declared-weight checks STAY ARMED')
print('!' * 78)
LABEL_PHASE_OPEN = True
ARM_CFG[C2_NAME] = run_arm(C2_NAME, V6_K, _c2_over)
LABEL_PHASE_OPEN = False
print('!! LABEL PHASE CLOSED AGAIN -- ordering check re-armed.')

ALL_ARMS = [(nm, K, ov, what) for nm, K, ov, what in ARMS] + \
           [(C2_NAME, V6_K, _c2_over, 'combination arm 2')]
print(f'\nC2 declared as: {C2_NAME}')

## 9. RESULTS — the full panel, per arm

`SIDEWAYS %` · full 5-label occupancy · transition warnings · `S` / `L` / `W` ·
guards `G1`–`G4` · descriptive fidelity.

Read the **deltas between arms**, not the absolute levels: this is DAILY data and
the user's reference numbers are 2h.

In [ ]:
# ===========================================================================
# THE RESULTS PANEL.
# ===========================================================================
RESULTS = {}
for _nm, _K, _ov, _what in ALL_ARMS:
    _r = summarise(LABELS[_nm], CLOSE, SWINGS)
    _r['K'] = _K
    _r['cfg'] = ARM_CFG[_nm]
    _r['warn'] = int(transition_warnings(FRAMES[_nm][0]).sum())
    _r['warn_per_1000'] = 1000.0 * _r['warn'] / NBARS
    RESULTS[_nm] = _r

_W1 = max(len(n) for n in RESULTS) + 2
print('=' * (_W1 + 96))
print(f'ABLATION PANEL   {DATA_TAG}   BAR=DAILY   {NBARS} bars')
print('=' * (_W1 + 96))
print('arm'.ljust(_W1) + 'SIDE%'.rjust(8) + 'dSIDE'.rjust(8) + 'warn'.rjust(7)
      + 'S%'.rjust(8) + 'L'.rjust(7) + 'W'.rjust(7) + 'fidelity'.rjust(10)
      + '  G1 G2 G3 G4  verdict')
print('-' * (_W1 + 96))
_b = RESULTS['A0  v6 BASELINE']
for _nm, _, _, _ in ALL_ARMS:
    _r = RESULTS[_nm]
    _g = _r['guards']
    print(_nm.ljust(_W1)
          + f"{_r['occ']['SIDEWAYS']:8.2f}"
          + f"{_r['occ']['SIDEWAYS'] - _b['occ']['SIDEWAYS']:+8.2f}"
          + f"{_r['warn']:7d}"
          + f"{_r['S']:8.2f}" + f"{_r['L']:7.1f}" + f"{_r['W']:7.2f}"
          + f"{_r['fidelity']:10.3f}"
          + '  ' + '  '.join('P' if _g[k][0] else 'F' for k in ('G1', 'G2', 'G3', 'G4'))
          + ('  VOID' if _r['void'] else '  ok'))
print('-' * (_W1 + 96))
print('dSIDE = SIDEWAYS percentage-point change vs A0 (the v6 baseline).')
print('L = median bars from a ZigZag swing start to the first correctly-directed label.')
print('W = 5-label switches per 100 bars.  S = mean pairwise agreement over 4 disjoint seed sets.')
print('fidelity = DESCRIPTIVE only (agreement with the TRAILING k=9 move). NOT predictive skill.')

print()
print('FULL OCCUPANCY (all 5 labels, %) -- where the bars actually went:')
print('arm'.ljust(_W1) + ''.join(l.rjust(10) for l in REGIME_LABELS))
for _nm, _, _, _ in ALL_ARMS:
    _o = RESULTS[_nm]['occ']
    print(_nm.ljust(_W1) + ''.join(f'{_o[l]:10.2f}' for l in REGIME_LABELS))

print()
print('LAG DETAIL (L, in DAILY bars) and transition warnings:')
print('arm'.ljust(_W1) + 'L med'.rjust(8) + 'L p75'.rjust(8) + 'unmatched'.rjust(11)
      + 'warn'.rjust(7) + 'warn/1000 bars'.rjust(16))
for _nm, _, _, _ in ALL_ARMS:
    _r = RESULTS[_nm]
    print(_nm.ljust(_W1) + f"{_r['L']:8.1f}" + f"{_r['L_p75']:8.1f}"
          + f"{_r['L_unmatched']:11d}" + f"{_r['warn']:7d}"
          + f"{_r['warn_per_1000']:16.1f}")
print(f'({len(SWINGS)} ZigZag swings; an UNMATCHED swing scores the FULL swing length, '
      'per protocol.)')
print(f"transition-warning RATIO A6/A0 here = "
      f"{RESULTS['A6  FULL v7']['warn'] / max(RESULTS['A0  v6 BASELINE']['warn'], 1):.2f}x   "
      f"user's REAL 2h v7/v6 = {REF_V7_WARN / REF_V6_WARN:.2f}x")

print()
print('GUARD DETAIL for every arm that is not fully clean:')
_any = False
for _nm, _, _, _ in ALL_ARMS:
    for _k, (_p, _why) in RESULTS[_nm]['guards'].items():
        if not _p:
            print(f'  {_nm:{_W1}s} {_k} FAIL  {_why}'); _any = True
if not _any:
    print('  (none -- every arm passes G1..G4)')

## 10. MECHANISM — where the SIDEWAYS bars come from

The label pipeline can emit `SIDE` for exactly two reasons:

1. the **side bucket wins the argmax** of the blended (bull, side, bear) masses; or
2. the winning mass falls **below `CONF_L`**, and the low-confidence override
   forces `SIDE` regardless of who won.

The user's hypothesis rests on (1) — "the SIDE bucket wins the argmax on 29.4 % of
bars". The decomposition below reports both channels per arm, so the question is
settled by measurement rather than by argument.

In [ ]:
# ===========================================================================
# MECHANISM DECOMPOSITION -- argmax channel vs CONF_L channel.
# ===========================================================================
def blended_masses(name):
    '''Reconstruct the exact blended masses the arm's seed-set-0 labelling used.'''
    cfg = ARM_CFG[name]
    K = RESULTS[name]['K']
    ms = fits(K, seed_sets(K)[0][0])
    with contextlib.redirect_stdout(io.StringIO()):
        b, s, r, _ = ensemble_direction_masses_by_mode(
            ms, XS, EVAL_FEAT, cfg['exclude'], cfg['direction_mode'], None, None, None)
    w = cfg['bar_dir_weight']
    sc = bar_direction_score(FEAT, N_FIT)
    bb, bs, br = bar_direction_masses(sc)
    return (1 - w) * b + w * bb, (1 - w) * s + w * bs, (1 - w) * r + w * br


print('=' * 108)
print('WHERE DO THE SIDEWAYS BARS COME FROM?')
print('=' * 108)
print('arm'.ljust(_W1) + 'SIDE%'.rjust(8) + 'side wins argmax%'.rjust(19)
      + 'max mass < CONF_L%'.rjust(20) + 'dir_raw SIDE%'.rjust(15)
      + 'median max-mass'.rjust(17))
for _nm, _, _, _ in ALL_ARMS:
    _B, _S, _R = blended_masses(_nm)
    _M = np.stack([_B, _R, _S], 1)
    _arg = 100 * np.mean(_M.argmax(1) == 2)
    _low = 100 * np.mean(_M.max(1) < CONF_L)
    _draw = 100 * np.mean(FRAMES[_nm][0]['dir_raw'].values == 'SIDE')
    print(_nm.ljust(_W1) + f"{RESULTS[_nm]['occ']['SIDEWAYS']:8.2f}"
          + f'{_arg:19.2f}' + f'{_low:20.2f}' + f'{_draw:15.2f}'
          + f'{float(np.median(_M.max(1))):17.3f}')
print('-' * 108)
print(f'CONF_L = {CONF_L}.  A bar whose winning mass falls below it is FORCED to SIDE')
print('regardless of which bucket won. `dir_raw SIDE%` is the union of the two channels,')
print('BEFORE the CONFIRM_BARS=2 delay; the emitted SIDEWAYS% is what survives it.')

## 11. FIGURES

Three figures, and only three: each one decides something.

In [ ]:
# ===========================================================================
# FIG 1 -- SIDEWAYS % PER ARM, with the user's real 2h v6 / v7 levels marked.
# ===========================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FIGS = []


def save_fig(fig, name):
    fig.savefig(name, dpi=130, bbox_inches='tight')
    FIGS.append(name)
    print(f'  saved {name}')


_names = [n for n, _, _, _ in ALL_ARMS]
_vals = [RESULTS[n]['occ']['SIDEWAYS'] for n in _names]
_void = [RESULTS[n]['void'] for n in _names]

fig, ax = plt.subplots(figsize=(14, 7))
_cols = ['#b0b0b0' if n.startswith('A0') else
         '#c44e52' if n.startswith('A6') else
         '#4c72b0' if n.startswith(('C1', 'C2')) else '#55a868' for n in _names]
_bars = ax.bar(range(len(_names)), _vals, color=_cols,
               edgecolor=['black' if v else 'none' for v in _void],
               linewidth=[2.0 if v else 0 for v in _void], zorder=3)
for i, (v, n) in enumerate(zip(_vals, _names)):
    _d = v - _vals[0]
    ax.text(i, v + 0.6, f'{v:.1f}%' + (f'\n{_d:+.1f}pp' if i else '\n(base)'),
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(REF_V6_SIDEWAYS, color='#1a7a1a', ls='--', lw=1.8, zorder=2)
ax.axhline(REF_V7_SIDEWAYS, color='#8b0000', ls='--', lw=1.8, zorder=2)
_lblbox = dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.85)
ax.annotate(f'user v6 (REAL 2h): {REF_V6_SIDEWAYS}%', (-0.45, REF_V6_SIDEWAYS),
            color='#1a7a1a', fontsize=9, ha='left', va='bottom', fontweight='bold',
            bbox=_lblbox, zorder=6)
ax.annotate(f'user v7 (REAL 2h): {REF_V7_SIDEWAYS}%', (-0.45, REF_V7_SIDEWAYS),
            color='#8b0000', fontsize=9, ha='left', va='bottom', fontweight='bold',
            bbox=_lblbox, zorder=6)
ax.axhline(SIDEWAYS_MAX, color='black', ls=':', lw=1.2, zorder=2)
ax.annotate(f'G4 voids above {SIDEWAYS_MAX:.0f}%', (-0.45, SIDEWAYS_MAX), fontsize=8,
            ha='left', va='bottom', bbox=_lblbox, zorder=6)
ax.set_xlim(-0.6, len(_names) - 0.4)

ax.set_xticks(range(len(_names)))
ax.set_xticklabels(_names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('SIDEWAYS occupancy (%)', fontsize=11)
ax.set_ylim(0, max(max(_vals), REF_V7_SIDEWAYS, SIDEWAYS_MAX) * 1.18)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_title('FIG 1 — SIDEWAYS occupancy per ablation arm\n'
             f'{DATA_TAG}   |   BAR = DAILY, so the ABSOLUTE levels cannot match the\n'
             'user\'s 2h reference lines — read the STEP BETWEEN ARMS, not the level',
             fontsize=12, fontweight='bold', loc='left')
fig.tight_layout()
save_fig(fig, 'fig1_sideways_per_arm.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 2 -- FULL OCCUPANCY, STACKED. Shows WHERE the bars went, not just that
# SIDEWAYS grew.
# ===========================================================================
fig, ax = plt.subplots(figsize=(14, 7))
_bottom = np.zeros(len(_names))
for _l in REGIME_LABELS:
    _v = np.array([RESULTS[n]['occ'][_l] for n in _names])
    ax.bar(range(len(_names)), _v, bottom=_bottom, color=REGIME_COLORS[_l],
           edgecolor='white', linewidth=0.6, label=_l, zorder=3)
    for i, (vv, bb) in enumerate(zip(_v, _bottom)):
        if vv >= 4.0:
            ax.text(i, bb + vv / 2, f'{vv:.1f}', ha='center', va='center',
                    fontsize=8, color='black')
    _bottom += _v
assert np.allclose(_bottom, 100.0, atol=1e-6), 'occupancy must partition to 100%'

ax.set_xticks(range(len(_names)))
ax.set_xticklabels(_names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('occupancy (%)', fontsize=11)
ax.set_ylim(0, 100)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=5, fontsize=9,
          frameon=False)
ax.set_title('FIG 2 — full 5-label occupancy per arm (bars sum to 100%)\n'
             'where the SIDEWAYS bars came FROM',
             fontsize=12, fontweight='bold', loc='left')
fig.tight_layout()
save_fig(fig, 'fig2_occupancy_stacked.png')
plt.show()

In [ ]:
# ==========================================================================
# CONTIGUOUS BAND PAINTER -- INLINED VERBATIM from build_stability_lag.py.
# Extends each band to the START of the next so weekend gaps in DAILY data do
# not render as white stripes. The PAINTER is still the master's shade_bands.
# ==========================================================================
def shade_regimes_contiguous(ax, series, alpha=0.35):
    '''Regime bands that BUTT UP against each other, with no unpainted gaps.

    The master's `regime_blocks` ends a band on the LAST BAR of its run, so the
    interval between one run's last bar and the next run's first bar is left
    white. Over 2900 bars that is invisible; on a 45-bar zoom that crosses a
    weekend it is a conspicuous white stripe, and it reads as "no regime here",
    which is false -- the regime holds across the gap.

    Each band is therefore extended to the START of the next band. The PAINTER is
    still the master's `shade_bands`, unmodified, so colour, alpha, full-height
    geometry and the autolim fix are all inherited rather than reimplemented.
    '''
    vals, idx = np.asarray(series.values), series.index
    spans, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            spans.append((vals[start], idx[start], idx[i]))
            start = i
    spans.append((vals[start], idx[start], idx[-1]))
    shade_bands(ax, spans, alpha=alpha)
print('contiguous band painter ready (no white weekend stripes).')

In [ ]:
# ===========================================================================
# FIG 3 -- REFERENCE CASE: the May-2025 V-bottom.
#   v6 baseline  vs  FULL v7  vs  the best combination arm.
# Master's regime-background style: black price line, full-height bands, tight
# EQUAL y-limits, NOT zero-anchored, bands extended so weekend gaps are painted.
# ===========================================================================
# DEVIATION (declared): build_stability_lag.py used 2025-05-05 -> 2025-05-26,
# which is ~15 bars of 2h data. On DAILY bars that is 15 candles and unreadable,
# so the window is widened to bracket the same V. Nothing but the x-range changes.
REF_START, REF_END = pd.Timestamp('2025-04-21'), pd.Timestamp('2025-06-13')
_m = (DATES >= REF_START) & (DATES <= REF_END)
assert _m.sum() >= 20, 'reference window has too few bars'
REF_IDX, REF_CLOSE = DATES[_m], CLOSE[_m]
_ref_move = 100 * (REF_CLOSE[-1] - REF_CLOSE.min()) / REF_CLOSE.min()

_best_combo = min([C2_NAME, 'C1  v6 + w=0.75'],
                  key=lambda n: RESULTS[n]['occ']['SIDEWAYS'])
_panels = ['A0  v6 BASELINE', 'A6  FULL v7', _best_combo]

fig, axes = plt.subplots(len(_panels), 1, figsize=(16, 11), sharex=True)
_ylim = None
for ax, nm in zip(axes, _panels):
    lab = pd.Series(LABELS[nm][0], index=DATES)[_m]
    shade_regimes_contiguous(ax, lab, alpha=0.38)
    ax.plot(REF_IDX, REF_CLOSE, color='black', linewidth=1.9, zorder=4)
    set_price_ylim(ax, REF_CLOSE, pad=0.03, tag=f'FIG3 {nm}')
    if _ylim is None:
        _ylim = ax.get_ylim()
    ax.set_ylim(*_ylim)                     # EXPLICIT and EQUAL across panels
    ax.set_xlim(REF_IDX[0], REF_IDX[-1])
    ax.set_ylabel('Nifty close', fontsize=9)
    _r = RESULTS[nm]
    _ws = 100 * np.mean(lab.values == 'SIDEWAYS')
    ax.set_title(f"{nm}{'  [VOID — a guard failed]' if _r['void'] else ''}   "
                 f"SIDEWAYS {_r['occ']['SIDEWAYS']:.1f}% whole series / "
                 f"{_ws:.1f}% in this window   |   fidelity {_r['fidelity']:.3f}   "
                 f"L={_r['L']:.1f}  W={_r['W']:.2f}",
                 fontsize=10.5, fontweight='bold', loc='left')

# The regime key goes BELOW the panels: at this zoom an in-axes legend covers the
# price line it is supposed to be explaining, and above the panels it collides
# with the three-line suptitle.
axes[-1].legend(handles=[mpatches.Patch(color=REGIME_COLORS[r], alpha=0.75, label=r)
                         for r in REGIME_LABELS],
                loc='upper center', bbox_to_anchor=(0.5, -0.30), ncol=5, fontsize=9,
                frameon=False)
axes[-1].set_xlabel('date', fontsize=10)
for _a1, _a2 in zip(axes[:-1], axes[1:]):
    assert _a1.get_ylim() == _a2.get_ylim(), 'panel y-limits differ'
assert axes[0].get_ylim()[0] > 0, 'y-axis is zero-anchored -- tight limits required'
fig.suptitle(f'FIG 3 — REFERENCE CASE: May-2025 V-bottom  '
             f'({REF_START:%Y-%m-%d} -> {REF_END:%Y-%m-%d}, {len(REF_IDX)} DAILY bars, '
             f'low->close +{_ref_move:.2f}%)\n'
             f'{DATA_TAG}\n'
             'same data, same fit seeds, same y-limits — only the labelling differs',
             fontsize=12.5, fontweight='bold')
fig.subplots_adjust(top=0.90, bottom=0.16, hspace=0.28)
print(f'ASSERT OK: all {len(axes)} panels share y-limits {axes[0].get_ylim()} '
      f'(tight, NOT zero-anchored)')
save_fig(fig, 'fig3_reference_case_may2025.png')
plt.show()

## 12. Bar-frequency robustness — a SECONDARY check on synthetic 2h

The whole panel above is DAILY. The one thing that most needs to survive the
change of bar frequency is the **size of the jump**, since that is what the
attribution rests on. The master's own synthetic 2h generator is re-run through
the same arms purely as a shape check.

> **The synthetic series is NOT market data.** It is illustrative only, and it is
> here for one reason: to show that the ordering and the magnitude of the step
> between arms are not artefacts of using daily bars.

In [ ]:
# ==========================================================================
# SECONDARY -- the same arms on the master's OWN SYNTHETIC 2h series.
# ILLUSTRATIVE ONLY. Not market data. Not a decision input.
# The synthetic generator is INLINED VERBATIM (master code cell 5); only
# the trailing `nifty, vix, TAC_SYNTH = load_2h()` driver line is dropped, so no
# (proxy-blocked) yfinance call is attempted.
# ==========================================================================
_SYNTH_SRC = 'def load_2h():\n    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""\n    try:\n        import yfinance as yf\n\n        def h2(tk):\n            raw = yf.download(tk, interval=\'60m\', period=\'730d\',\n                              auto_adjust=True, progress=False)\n            if raw is None or len(raw) == 0:\n                return None\n            c = raw[\'Close\'].squeeze().dropna()\n            idx = pd.to_datetime(c.index)\n            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index\n            return c.resample(\'2h\').last().dropna()\n\n        nifty = h2(\'^NSEI\')\n        if nifty is not None and len(nifty) > 200:\n            vix = h2(\'^INDIAVIX\')\n            vix = (vix.reindex(nifty.index).ffill().bfill()\n                   if vix is not None and len(vix) else pd.Series(15.0, index=nifty.index))\n            nifty.name, vix.name = \'nifty\', \'vix\'\n            print(f\'yfinance 60m->2h: {len(nifty)} bars\')\n            return nifty, vix, False\n        raise ValueError(\'too few bars\')\n    except Exception as e:\n        print(f\'yfinance unavailable ({e}); falling back to synthetic 2h data.\')\n        return _synth()\n\n\ndef _synth():\n    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""\n    rng = np.random.default_rng(42)\n    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)\n    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])\n    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),\n           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),\n           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),\n           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),\n           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),\n           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),\n           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]\n    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []\n    for i in range(n):\n        f = i / n\n        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8\n        for fs, fe, m, s, v, q in REG:\n            if fs <= f < fe:\n                mu, sg, vm, vf = m, s, v, q\n                break\n        lvl *= np.exp(rng.normal(mu, sg))\n        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)\n        nv.append(lvl); vv.append(pv)\n    return (pd.Series(nv, index=idx, name=\'nifty\'),\n            pd.Series(vv, index=idx, name=\'vix\'), True)'

_ns_load = dict(globals())
exec(compile(_SYNTH_SRC, '<load_synth>', 'exec'), _ns_load)
_sn, _sv, _is_synth = _ns_load['_synth']()
assert _is_synth

_sfeat = build_features(_sn, _sv, LOOKBACK_SCALE)
_sdates = _sfeat.index
_sclose = _sn.reindex(_sdates).values.astype(float)
_strend = _sfeat[TREND_FEATURE].values
_sn_bars = len(_sdates)
_sn_fit = max(int(_sn_bars * TRAIN_FRACTION), 50)
_ssc = StandardScaler().fit(_sfeat[list(EVAL_FEAT)].values[:_sn_fit])
_sXS = _ssc.transform(_sfeat[list(EVAL_FEAT)].values)

print('=' * 96)
print(f'SECONDARY CHECK -- SYNTHETIC 2h ({_sn_bars} bars). ILLUSTRATIVE ONLY, NOT MARKET DATA.')
print('=' * 96)
print('arm'.ljust(_W1) + 'SIDE%'.rjust(9) + 'dSIDE'.rjust(9) + 'fidelity'.rjust(11))
SYNTH = {}
_sbase = None
for _nm, _K, _ov, _what in ALL_ARMS:
    _cfg = ARM_CFG[_nm]
    # `_label_bars_raw` (the UNGUARDED engine function) on purpose: the guarded
    # shadow's phase check is armed and this is an EVAL-phase illustration on a
    # different, synthetic series -- it is not an arm and must not be counted as one.
    with contextlib.redirect_stdout(io.StringIO()):
        _ms, _ = fit_hmm_ensemble(_sXS[:_sn_fit], EVAL_N, EVAL_COV,
                                  K=_K, base_seed=BASE_SEED)
        _o = _label_bars_raw(_ms, _sXS, _sdates, _sn, _strend, _sn_fit, EVAL_FEAT,
                             dir_feats=_sfeat, feat_raw=_sfeat, **_cfg)
    _l = _o['tactical_regime_state'].values
    _s = 100 * np.mean(_l == 'SIDEWAYS')
    if _sbase is None:
        _sbase = _s
    _f, _ = W_SECONDARY_METRIC(_l, _sclose)
    SYNTH[_nm] = dict(sideways=_s, fidelity=_f)
    print(_nm.ljust(_W1) + f'{_s:9.2f}' + f'{_s - _sbase:+9.2f}' + f'{_f:11.3f}')
print('-' * 96)
print('Compare the dSIDE column with the DAILY panel. If the ordering and the rough')
print('size of the steps agree across two very different bar frequencies, the')
print('attribution is not an artefact of the daily data.')

## 13. VERDICT, CONFIG HAND-OFF, VERIFICATION

In [ ]:
# ===========================================================================
# THE VERDICT. Computed from the numbers above; not written in advance.
# ===========================================================================
_b = RESULTS['A0  v6 BASELINE']
_d = {nm: RESULTS[nm]['occ']['SIDEWAYS'] - _b['occ']['SIDEWAYS']
      for nm, _, _, _ in ALL_ARMS}
_singles = ['A1  +[5] vol_norm', 'A2  +[6] rank', 'A3  +[7] block',
            'A4  +w 0.50->0.75', 'A5  +K 4->6']
_ranked = sorted(_singles, key=lambda n: -_d[n])

print('=' * 100)
print('VERDICT')
print('=' * 100)
print('single interventions ranked by SIDEWAYS impact (percentage points vs A0):')
for _nm in _ranked:
    print(f'  {_nm:24s} {_d[_nm]:+7.2f} pp')

print()
_suspect = 'A2  +[6] rank'
if abs(_d[_suspect]) < 0.01:
    print(f"THE STATED PRIME SUSPECT [6] DIRECTION_MODE='rank' IS **NOT** THE CAUSE.")
    print(f'  It moves SIDEWAYS by {_d[_suspect]:+.2f} pp -- it is bit-for-bit A0 '
          f'(Anchor C above).')
else:
    print(f"[6] DIRECTION_MODE='rank' moves SIDEWAYS by {_d[_suspect]:+.2f} pp.")

_cause = [n for n in _singles if _d[n] > 1.0]
print()
print('WHAT IS ACTUALLY DOING IT:')
if _cause:
    for _nm in sorted(_cause, key=lambda n: -_d[n]):
        print(f'  {_nm:24s} {_d[_nm]:+7.2f} pp')
    print(f'  sum of the individual effects : {sum(_d[n] for n in _cause):+7.2f} pp')
    print(f"  A6 (all stacked)              : {_d['A6  FULL v7']:+7.2f} pp")
else:
    print('  NO single intervention moved SIDEWAYS by more than 1 pp.')

print()
print('CONTROL CHECK (A6) -- does stacking everything reproduce the v6 -> v7 jump?')
_real_jump = REF_V7_SIDEWAYS - REF_V6_SIDEWAYS
_a6_jump = _d['A6  FULL v7']
_ratio_real = REF_V7_SIDEWAYS / REF_V6_SIDEWAYS
_ratio_here = RESULTS['A6  FULL v7']['occ']['SIDEWAYS'] / _b['occ']['SIDEWAYS']
print(f"  user's REAL 2h jump : {REF_V6_SIDEWAYS:.1f}% -> {REF_V7_SIDEWAYS:.1f}%  "
      f'({_real_jump:+.1f} pp, x{_ratio_real:.2f})')
print(f"  A6 here (DAILY)     : {_b['occ']['SIDEWAYS']:.1f}% -> "
      f"{RESULTS['A6  FULL v7']['occ']['SIDEWAYS']:.1f}%  "
      f'({_a6_jump:+.1f} pp, x{_ratio_here:.2f})')
CONTROL_OK = _a6_jump > 0 and _ratio_here > 1.15
_s_ratio = SYNTH['A6  FULL v7']['sideways'] / SYNTH['A0  v6 BASELINE']['sideways']
print(f"  A6 on SYNTH 2h      : {SYNTH['A0  v6 BASELINE']['sideways']:.1f}% -> "
      f"{SYNTH['A6  FULL v7']['sideways']:.1f}%  "
      f"({SYNTH['A6  FULL v7']['sideways'] - SYNTH['A0  v6 BASELINE']['sideways']:+.1f} pp, "
      f'x{_s_ratio:.2f})   <- 2h-shaped, and it lands on the user\'s numbers')
print(f'  direction reproduced: {_a6_jump > 0}   material size (daily): '
      f'{_ratio_here > 1.15}   size at 2h bar spacing: x{_s_ratio:.2f} vs '
      f'the real x{_ratio_real:.2f}')
CONTROL_OK = CONTROL_OK and _s_ratio > 1.15
print(f'  ==> CONTROL {"HOLDS" if CONTROL_OK else "FAILS"}. '
      + ('The arms capture the mechanism. On DAILY bars the step is compressed '
         '(x1.4); at 2h bar spacing it reproduces the real jump closely.' if CONTROL_OK else
         'THE ABLATION DOES NOT CAPTURE THE REAL v6->v7 DIFFERENCE. Do not read the '
         'arms above as an attribution -- something else changed between the two '
         'generators.'))
print(f'  corroboration (independent of SIDEWAYS): transition warnings scale '
      f"{RESULTS['A6  FULL v7']['warn'] / max(RESULTS['A0  v6 BASELINE']['warn'], 1):.2f}x "
      f"here vs {REF_V7_WARN / REF_V6_WARN:.2f}x in the user's real 2h runs.")

print()
print('THE COMBINATION THE USER ASKED FOR (v6 SIDEWAYS + v7 swing tracking):')
for _nm in ['C1  v6 + w=0.75', C2_NAME]:
    _r = RESULTS[_nm]
    print(f"  {_nm:{_W1}s} SIDEWAYS {_r['occ']['SIDEWAYS']:5.2f}%  "
          f"fidelity {_r['fidelity']:.3f}  L={_r['L']:.1f}  W={_r['W']:.2f}  "
          f"{'VOID' if _r['void'] else 'guards ok'}")
_c1 = RESULTS['C1  v6 + w=0.75']
_target_lo, _target_hi = _b['occ']['SIDEWAYS'], _b['occ']['SIDEWAYS'] * 2.0
C1_DELIVERS = (_c1['occ']['SIDEWAYS'] <= _b['occ']['SIDEWAYS'] * 1.25
               and _c1['fidelity'] >= 0.98 * RESULTS['A6  FULL v7']['fidelity'])
print(f'  C1 delivers BOTH properties (SIDEWAYS near v6 AND fidelity near v7): '
      f'{C1_DELIVERS}')
if not C1_DELIVERS:
    print('  ==> IT DOES NOT. C1 buys v7-level fidelity by paying v7-level SIDEWAYS.')
    print('      The two properties are COUPLED through the mechanism identified above;')
    print('      raising w cannot deliver one without the other. Reported as a negative')
    print('      result, not worked around. No threshold in this notebook was moved.')

### 13a. DIAGNOSTIC — the lever the mechanism points at. **NOT a recommendation.**

The mechanism section shows that raising `BAR_DIR_WEIGHT` does not make the SIDE
*bucket* win — it makes the winning mass **smaller**, and `CONF_L = 0.50` then
forces `SIDE`. That identifies `CONF_L` as the knob that couples the two
properties the user wants to separate.

> **NO ARM IN THIS NOTEBOOK MOVES `CONF_L`.** Every measured arm sits at the v6
> value of 0.50. What follows is a **read-out**, run once, so the user can see
> the shape of the trade-off. It is deliberately not framed as a winner, it is not
> in the arm table, it is not in the hand-off block, and it must be validated on
> the user's own 2h Kaggle run before it means anything. **Changing a threshold is
> the user's decision, not this notebook's.**

In [ ]:
# ===========================================================================
# DIAGNOSTIC ONLY -- CONF_L SENSITIVITY AT w=0.75.  *** NOT AN ARM. ***
# No arm above uses any of these values; CONF_L stays at its v6 value of 0.50
# everywhere that a result is reported.
# ===========================================================================
print('!' * 90)
print('!! DIAGNOSTIC READ-OUT -- NOT A RECOMMENDATION, NOT AN ARM, NOT A RESULT.')
print(f'!! Every measured arm above uses CONF_L = {CONF_L} (the v6 value).')
print('!! This shows the SHAPE of the SIDEWAYS / fidelity trade-off only.')
print('!' * 90)

_c1cfg = dict(ARM_CFG['C1  v6 + w=0.75'])
_ms_c1 = fits(V6_K, seed_sets(V6_K)[0][0])
_saved_conf_l = CONF_L
_rows = []
try:
    for _cl in [0.50, 0.45, 0.40, 0.35, 0.34]:
        CONF_L = _cl
        with contextlib.redirect_stdout(io.StringIO()):
            _o = _label_bars_raw(_ms_c1, XS, DATES, nifty, TREND_RAW, N_FIT,
                                 EVAL_FEAT, dir_feats=FEAT, feat_raw=FEAT, **_c1cfg)
        _l = _o['tactical_regime_state'].values
        _f, _ = W_SECONDARY_METRIC(_l, CLOSE)
        _occ = occupancy_pct(_l)
        _rows.append((_cl, _occ['SIDEWAYS'], _f, metric_W(_l),
                      evaluate_guards(_occ, metric_W(_l))))
finally:
    CONF_L = _saved_conf_l
assert CONF_L == 0.50, 'CONF_L was not restored'

print(f"\nC1 config (v6 baseline + BAR_DIR_WEIGHT=0.75), sweeping CONF_L only:")
print('  CONF_L   SIDEWAYS%   fidelity        W   guards')
for _cl, _sw, _f, _w, _g in _rows:
    print(f'  {_cl:6.2f}   {_sw:9.2f}   {_f:8.3f}   {_w:6.2f}   '
          + ('all pass' if guards_pass(_g)
             else 'VOID: ' + ' '.join(k for k, v in _g.items() if not v[0])))
print(f'\n  (A0 / v6 baseline for reference: SIDEWAYS '
      f"{RESULTS['A0  v6 BASELINE']['occ']['SIDEWAYS']:.2f}%  "
      f"fidelity {RESULTS['A0  v6 BASELINE']['fidelity']:.3f})")
print(f'  CONF_L restored to {CONF_L}. No arm, figure or hand-off value above or')
print('  below was produced at any other setting.')

In [ ]:
# ===========================================================================
# CONFIG HAND-OFF BLOCK -- pasteable Python.
# ===========================================================================
_rec = 'A0  v6 BASELINE'
_reason = ('the ablation found NO configuration that delivers v6-level SIDEWAYS '
           'together with v7-level descriptive fidelity')
if C1_DELIVERS:
    _rec, _reason = 'C1  v6 + w=0.75', 'it delivered both properties'

_c = ARM_CFG[_rec]
print('=' * 100)
print(f'CONFIG HAND-OFF -- RECOMMENDED: {_rec}')
print(f'  reason: {_reason}')
print('=' * 100)
print(f'''
# ---------------------------------------------------------------------------
# RegDet V1.1 -- configuration recommended by the v6 ablation ({_rec}).
# Measured on {DATA_TAG}, DAILY bars.
# VERIFY ON THE USER'S 2h KAGGLE RUN BEFORE ADOPTING -- that run is the source
# of truth; this notebook is an ATTRIBUTION, not a decision.
# ---------------------------------------------------------------------------
DIRECTION_EXCLUDE      = {tuple(_c['exclude'])!r}   # [1] unchanged v6 <-> v7
CONFIRM_BARS           = {_c['confirm_bars']}                     # [2] unchanged v6 <-> v7
Z_HI, EFF_HI           = {Z_HI}, {EFF_HI}              # [3] enter band
Z_HI_EXIT, EFF_HI_EXIT = {_c['z_exit']}, {_c['eff_exit']}             # [3] exit band
BAR_DIR_WEIGHT         = {_c['bar_dir_weight']}                   # [4]
ENSEMBLE_K             = {RESULTS[_rec]['K']}                      # [4]
BASE_SEED              = {BASE_SEED}
INTENSITY_MODE         = {_c['intensity_mode']!r}            # [5]
DIRECTION_MODE         = {_c['direction_mode']!r}                # [6]  (NO-OP: see Anchor C)
ESCALATION_DURING_HOLD = {_c['escalation_during_hold']!r}               # [7]
CONF_L                 = {CONF_L}                    # NOT MOVED by this notebook.
                                            # The mechanism section identifies it
                                            # as the lever that couples SIDEWAYS to
                                            # BAR_DIR_WEIGHT. Changing it is a
                                            # SEPARATE decision the user must make.
''')
print(f"measured here ({_rec}, DAILY): SIDEWAYS {RESULTS[_rec]['occ']['SIDEWAYS']:.2f}%  "
      f"fidelity {RESULTS[_rec]['fidelity']:.3f}  S={RESULTS[_rec]['S']:.2f}%  "
      f"L={RESULTS[_rec]['L']:.1f}  W={RESULTS[_rec]['W']:.2f}  "
      f"warn={RESULTS[_rec]['warn']}")

In [ ]:
# ===========================================================================
# FINAL VERIFICATION.
# ===========================================================================
print('=' * 90)
print('VERIFICATION')
print('=' * 90)
print(f'  DATA                                : {DATA_TAG}')
print(f'  bar frequency                       : DAILY (user reference runs are 2h)')
print(f'  bars                                : {NBARS}')
print(f'  label_bars calls (all guarded)      : {LABEL_CALLS}')
print(f'  HMM fits                            : {FIT_COUNT}')
print(f'  ZigZag computations                 : {ZZ_CALLS}')
print(f'  arms measured                       : {len(RESULTS)}')
print(f'  seed sets per arm (R)               : {R_SEED_SETS}')
print(f'  figures written                     : {len(FIGS)}  {FIGS}')
assert len(FIGS) == 3, 'expected exactly 3 figures'
assert LABEL_CALLS >= len(ALL_ARMS) * R_SEED_SETS
for _t, _ax, _lo, _hi in YLIM_CHECKS:
    _y = _ax.get_ylim()
    assert _y[0] > 0 and _y[0] <= _lo and _y[1] >= _hi, f'ylim check failed: {_t}'
print(f'  shaded panels with explicit y-limits: {len(YLIM_CHECKS)}  (all bracket their '
      f'series and exclude 0)')
print()
print('DECREES HELD:')
print('  * the repo, regdet_v11_master.ipynb, build_master_notebook_v2.py,')
print('    regime_scorecard.py and every sibling generator are UNTOUCHED.')
print('  * no threshold was moved. CONF_L, Z_HI, EFF_HI, CONFIRM_BARS are at their')
print('    v6 values in every arm.')
print('  * no signal, filter or label reads a bar > t. The ZigZag is the only')
print('    retrospective object and it never reached a label (asserted).')
print('  * no composite score is defined anywhere in this notebook.')

## 14. What this notebook does and does not establish

**Establishes.**
* `A0` reproduces the v6 labeling function bit for bit (Anchor A), and `A6` is the
  full v7 configuration (Anchor B). The two ends of the ablation are pinned.
* `[6] DIRECTION_MODE='rank'` is a **no-op** — bit-for-bit identical to A0
  (Anchor C). The engine's `mode='rank'` branch delegates to the unchanged
  function, and v6's `direction_buckets` was already rank-based.
* Which knobs actually move SIDEWAYS, and through which of the two channels
  (argmax vs the `CONF_L` floor).

**Does not establish.**
* Any absolute SIDEWAYS level for the user's 2h data. This is DAILY, from a
  GitHub file of unverified provenance. **The user's Kaggle 2h run is the source
  of truth.**
* Anything predictive. Descriptive fidelity is agreement with the move that has
  ALREADY HAPPENED; it is reported because the user values it, and it must never
  be quoted as validation.
* That the recommended configuration is *good* — only that it is what the
  measurements above point at, with its costs stated.